# 05 — SRE Toolkit

The day-to-day modules: running shell commands, env vars, dates/times, HTTP calls, retries, logging, and a small CLI scaffold.

## Running shell commands (`subprocess`)

In [ ]:
import subprocess

# run a command, capture output
result = subprocess.run(
    ["echo", "hello from shell"],
    capture_output=True,
    text=True,          # decode bytes -> str
)
print("stdout:", result.stdout.strip())
print("returncode:", result.returncode)

# check=True raises CalledProcessError on non-zero exit
try:
    subprocess.run(["ls", "/does-not-exist"], capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as e:
    print("failed:", e.returncode, e.stderr.strip())

# a real command (uname is on mac/linux)
out = subprocess.run(["uname", "-a"], capture_output=True, text=True).stdout
print(out.strip())

**Tip:** prefer a list of args (`["ls", "-l"]`) over a string. Only use `shell=True` when you truly need shell features — and never with untrusted input (injection risk).

## OS, environment, arguments (`os`, `sys`)

In [ ]:
import os
import sys

# environment variables
print(os.environ.get("HOME"))
print(os.environ.get("MISSING", "default-value"))   # safe default
os.environ["MY_FLAG"] = "1"                            # set for this process

print(os.getcwd())            # current dir
print(sys.platform)           # 'darwin', 'linux', 'win32'
print(sys.version.split()[0]) # python version

# command-line args (in a real script): sys.argv[0] is the script name
print("args would be:", sys.argv)

# exit a script with a status code
# sys.exit(1)   # non-zero = failure (uncomment in a real script)

## Dates & times (`datetime`)

In [ ]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)
print(now.isoformat())
print(now.strftime("%Y-%m-%d %H:%M:%S"))   # format to string

# parse a string -> datetime
ts = datetime.strptime("2026-06-15 10:00:02", "%Y-%m-%d %H:%M:%S")
print(ts)

# arithmetic with timedelta
an_hour_ago = now - timedelta(hours=1)
print("1h ago:", an_hour_ago.strftime("%H:%M:%S"))

# difference between two times
delta = now - ts.replace(tzinfo=timezone.utc)
print("seconds elapsed:", delta.total_seconds())

# unix epoch <-> datetime
print("epoch:", int(now.timestamp()))
print(datetime.fromtimestamp(1718445600, timezone.utc))

## Timing how long something takes

In [ ]:
import time

start = time.perf_counter()
time.sleep(0.2)                    # pretend to do work
elapsed = time.perf_counter() - start
print(f"took {elapsed:.3f}s")

## HTTP requests (health checks, APIs)

`requests` is the standard (`pip install requests`). Falls back to the stdlib `urllib` if not installed.

In [ ]:
try:
    import requests

    r = requests.get("https://httpbin.org/json", timeout=5)   # ALWAYS set a timeout
    print("status:", r.status_code)
    print("ok?", r.ok)
    data = r.json()                # parse JSON response
    print(data)

    # POST with JSON body + headers
    # r = requests.post(url, json={"k": "v"}, headers={"Authorization": "Bearer ..."}, timeout=5)
    # r.raise_for_status()         # raise on 4xx/5xx
except Exception as e:
    print("request skipped/failed (no network?):", e)

In [ ]:
# stdlib fallback — no install needed
import urllib.request
import json

try:
    req = urllib.request.Request("https://httpbin.org/json")
    with urllib.request.urlopen(req, timeout=5) as resp:
        print("status:", resp.status)
        body = json.loads(resp.read())
        print(body)
except Exception as e:
    print("urllib skipped/failed:", e)

## Retry with backoff (common SRE pattern)

In [ ]:
import time

def retry(func, attempts=3, base_delay=0.5):
    """Call func(), retrying with exponential backoff."""
    for i in range(attempts):
        try:
            return func()
        except Exception as e:
            if i == attempts - 1:
                raise               # last attempt: give up
            delay = base_delay * (2 ** i)   # 0.5, 1.0, 2.0...
            print(f"attempt {i+1} failed ({e}); retrying in {delay}s")
            time.sleep(delay)

# demo: fails twice then succeeds
state = {"n": 0}
def flaky():
    state["n"] += 1
    if state["n"] < 3:
        raise RuntimeError("temporary glitch")
    return "success"

print(retry(flaky))

## Logging (better than print for real scripts)

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("healthcheck")

log.debug("won't show (below INFO level)")
log.info("starting checks")
log.warning("cpu at 87%%")
log.error("disk full on /var")

# log an exception with traceback
try:
    1 / 0
except ZeroDivisionError:
    log.exception("math broke")

## CLI script scaffold (`argparse`)

This is what a real `.py` script (not a notebook) usually looks like.

In [ ]:
import argparse

def build_parser():
    p = argparse.ArgumentParser(description="Check host health")
    p.add_argument("host", help="hostname to check")          # positional
    p.add_argument("--threshold", type=int, default=80)        # optional w/ default
    p.add_argument("--verbose", action="store_true")           # flag
    return p

# simulate command line: ./check.py web-01 --threshold 90 --verbose
args = build_parser().parse_args(["web-01", "--threshold", "90", "--verbose"])
print(args.host, args.threshold, args.verbose)

# A real script ends with:
# if __name__ == "__main__":
#     args = build_parser().parse_args()
#     main(args)

## Putting it together: a tiny health-check function

In [ ]:
import subprocess
from datetime import datetime, timezone

def ping(host, count=1):
    """Return True if host responds to ping."""
    # -c on mac/linux; use ['-n', str(count)] on Windows
    result = subprocess.run(
        ["ping", "-c", str(count), host],
        capture_output=True, text=True,
    )
    return result.returncode == 0

for host in ["127.0.0.1", "localhost"]:
    up = ping(host)
    now = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"{now} {host:12} {'UP' if up else 'DOWN'}")